# ETL — VISp Inhibitory Patch-seq: DataSet & DataItem

Writes one `DataSet` record (`dataset_id = "visp_inh_patchseq"`, `project_id = "visp_patchseq"`), one `DataItem` per cell from `patchseq_tx_cell_ttype_labels.csv`, and the corresponding `DataItemDataSetAssociation` links. No prerequisites; features and cluster mappings are written in `_02` and `_03`.

In [1]:
import pandas as pd
import polars as pl
import pyarrow as pa

from connects_common_connectivity.models import (
    DataSet,
    DataItem,
    DataItemDataSetAssociation,
    Modality,
)
from connects_common_connectivity.config import output_root
from connects_common_connectivity.io import write_models


In [2]:
INPUT_CSV  = "/data/visp-features-and-mapping/patchseq_tx_cell_ttype_labels.csv"
OUTPUT_ROOT = output_root()
PROJECT_ID  = "visp_patchseq"
DATASET_ID  = "visp_inh_patchseq"

print(f"INPUT_CSV   : {INPUT_CSV}")
print(f"OUTPUT_ROOT : {OUTPUT_ROOT}")
print(f"PROJECT_ID  : {PROJECT_ID}")
print(f"DATASET_ID  : {DATASET_ID}")

INPUT_CSV   : /data/visp-features-and-mapping/patchseq_tx_cell_ttype_labels.csv
OUTPUT_ROOT : ../scratch/em_patchseq_wnm_v2/
PROJECT_ID  : visp_patchseq
DATASET_ID  : visp_inh_patchseq


## Load input CSV

In [3]:
df = pd.read_csv(INPUT_CSV, index_col="spec_id_label", dtype={"spec_id_label": str})
print("Shape:", df.shape)
df.head(3)

Shape: (2759, 1)


,ttype
spec_id_label,
888001481,Lamp5 Fam19a1 Tmem182
736493069,Lamp5 Fam19a1 Tmem182
830445950,Lamp5 Fam19a1 Tmem182


## Write `DataSet`

In [4]:
dataset = DataSet(
    id=DATASET_ID,
    name="VISp inhibitory Patch-seq data set",
    publication="doi.org/10.1016/j.cell.2020.09.057",
    modality=Modality.MORPHOLOGY.value,
    project_id=PROJECT_ID,
)
result = write_models([dataset])
print(f"DataSet written: {result.rows_written} rows")

DataSet written: 1 rows


In [5]:
# Verification
ds_verify = (
    pl.read_delta(OUTPUT_ROOT + "dataset/")
    .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("id") == DATASET_ID))
)
print(ds_verify.shape)
print(ds_verify.head())
assert ds_verify.shape[0] == 1, f"Expected 1 DataSet row for {DATASET_ID}, got {ds_verify.shape[0]}"
assert ds_verify["id"][0] == DATASET_ID, "DataSet id mismatch"

(1, 5)
shape: (1, 5)
┌───────────────────┬─────────────────┬───────────────────────────────┬────────────┬───────────────┐
│ id                ┆ name            ┆ publication                   ┆ modality   ┆ project_id    │
│ ---               ┆ ---             ┆ ---                           ┆ ---        ┆ ---           │
│ str               ┆ str             ┆ str                           ┆ str        ┆ str           │
╞═══════════════════╪═════════════════╪═══════════════════════════════╪════════════╪═══════════════╡
│ visp_inh_patchseq ┆ VISp inhibitory ┆ doi.org/10.1016/j.cell.2020.0 ┆ MORPHOLOGY ┆ visp_patchseq │
│                   ┆ Patch-seq data… ┆ 9…                            ┆            ┆               │
└───────────────────┴─────────────────┴───────────────────────────────┴────────────┴───────────────┘


## Write `DataItem`

In [6]:
cell_ids = df.index.astype(str).tolist()

dataitems = [
    DataItem(id=cid, name=cid, project_id=PROJECT_ID)
    for cid in cell_ids
]
n_appended = write_models(dataitems).rows_written
print(f"DataItem rows appended: {n_appended} (total in batch: {len(cell_ids)})")

DataItem rows appended: 2759 (total in batch: 2759)


In [7]:
# Verification
di_verify = (
    pl.read_delta(OUTPUT_ROOT + "dataitem/")
    .filter(pl.col("project_id") == PROJECT_ID)
)
print(di_verify.shape)
print(di_verify.head())
registered_ids = set(di_verify["id"].to_list())
assert all(cid in registered_ids for cid in cell_ids), "Some cell_ids are missing from DataItem table"
assert di_verify["id"].n_unique() == di_verify.shape[0], "Duplicate DataItem ids detected"

(4287, 4)
shape: (5, 4)
┌───────────┬───────────┬───────────────────┬───────────────┐
│ id        ┆ name      ┆ neuroglancer_link ┆ project_id    │
│ ---       ┆ ---       ┆ ---               ┆ ---           │
│ str       ┆ str       ┆ str               ┆ str           │
╞═══════════╪═══════════╪═══════════════════╪═══════════════╡
│ 888001481 ┆ 888001481 ┆ null              ┆ visp_patchseq │
│ 736493069 ┆ 736493069 ┆ null              ┆ visp_patchseq │
│ 830445950 ┆ 830445950 ┆ null              ┆ visp_patchseq │
│ 644941196 ┆ 644941196 ┆ null              ┆ visp_patchseq │
│ 658075752 ┆ 658075752 ┆ null              ┆ visp_patchseq │
└───────────┴───────────┴───────────────────┴───────────────┘


## Write `DataItemDataSetAssociation`

In [8]:
associations = [
    DataItemDataSetAssociation(
        dataitem_id=cid,
        dataset_id=DATASET_ID,
        project_id=PROJECT_ID,
    )
    for cid in cell_ids
]
result = write_models(associations)
print(f"DataItemDataSetAssociation written: {result.rows_written} rows")

DataItemDataSetAssociation written: 2759 rows


In [9]:
# Verification
assoc_verify = (
    pl.read_delta(OUTPUT_ROOT + "dataitem_dataset_association/")
    .filter(
        (pl.col("project_id") == PROJECT_ID) & (pl.col("dataset_id") == DATASET_ID)
    )
)
print(assoc_verify.shape)
print(assoc_verify.head())
assert assoc_verify.shape[0] == len(cell_ids), (
    f"Expected {len(cell_ids)} association rows, got {assoc_verify.shape[0]}"
)
assert (assoc_verify["dataset_id"] == DATASET_ID).all(), "Not all associations point to DATASET_ID"

(2759, 3)
shape: (5, 3)
┌─────────────┬───────────────────┬───────────────┐
│ dataitem_id ┆ dataset_id        ┆ project_id    │
│ ---         ┆ ---               ┆ ---           │
│ str         ┆ str               ┆ str           │
╞═════════════╪═══════════════════╪═══════════════╡
│ 888001481   ┆ visp_inh_patchseq ┆ visp_patchseq │
│ 736493069   ┆ visp_inh_patchseq ┆ visp_patchseq │
│ 830445950   ┆ visp_inh_patchseq ┆ visp_patchseq │
│ 644941196   ┆ visp_inh_patchseq ┆ visp_patchseq │
│ 658075752   ┆ visp_inh_patchseq ┆ visp_patchseq │
└─────────────┴───────────────────┴───────────────┘


## Summary

| Output path | Class | Rows |
|---|---|---|
| `dataset/` | `DataSet` | 1 |
| `dataitem/` | `DataItem` | 2 759 |
| `dataitem_dataset_association/` | `DataItemDataSetAssociation` | 2 759 |

**Input columns intentionally not written here:**
- `ttype` — T-type label; written in a later notebook as `CellToClusterMapping`.